In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "matched_df.csv")
df = pd.read_csv(file_path)

In [ ]:
print(len(df))

In [ ]:
df.columns

# Standardize values


In [ ]:
# List of variables to standardize [degree type related]
dg_vars = ['D2DG', 'D3DG', 'D4DG', 'D5DG']

# Before conversion
print("Before conversion:")
for var in dg_vars:
    print(f"\n{var} unique values:", df[var].unique())

# Convert float values to strings without decimal points
for var in dg_vars:
    # Convert non-'L' values to integers (this will handle both '1.0' and '1' to 1)
    mask = (df[var] != 'L') & df[var].notna()
    df.loc[mask, var] = df.loc[mask, var].astype(float).astype(int).astype(str)
    
# After conversion
print("\nAfter conversion:")
for var in dg_vars:
    print(f"\n{var} value counts:")
    print(df[var].value_counts(dropna=False))

In [ ]:
print(df['DGRDG'].unique())

In [ ]:
df.columns

# Filtering

In [ ]:
# 1. valid cohort 'BAYR'
df = df[df['BAYR']< 9998.0]

print(f'# Obs valid cohort:{len(df)}')

# 2. necessary salary range

df = df[df['SALARY'] > 0]

print(f'# Obs positive salary:{len(df)}')


In [ ]:
df['SALARY'].describe()

# merge UR data

In [ ]:
# Define the path and read the CSV files
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")
ur_ma_path = os.path.join(base_path, "ur_ma_16y.csv")

# Read the files
ur_ba_16y = pd.read_csv(ur_ba_path)
ur_ma_16y = pd.read_csv(ur_ma_path)
# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})
ur_ma_16y = ur_ma_16y.rename(columns={'CGMD16O': 'UR'})

In [ ]:
ur_ma_16y

In [ ]:
# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year
ur_ba_16y_yearly = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
ur_ba_16y_yearly = ur_ba_16y_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ba'})

# For Master's
ur_ma_16y['observation_date'] = pd.to_datetime(ur_ma_16y['observation_date'])
ur_ma_16y['year'] = ur_ma_16y['observation_date'].dt.year
ur_ma_16y_yearly = ur_ma_16y.groupby('year')['UR'].mean().reset_index()
ur_ma_16y_yearly = ur_ma_16y_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ma'})

# Create separate dataframes for bachelor's and master's degree holders
df_ba = df[df['DGRDG'] == 1].copy()
df_ma = df[df['DGRDG'] == 2].copy()

# Merge unemployment rates
df_ba = df_ba.merge(ur_ba_16y_yearly, on='BAYR', how='left')
df_ma = df_ma.merge(ur_ma_16y_yearly, on='BAYR', how='left')

# Combine back and create final gradur column
df_ba['gradur'] = df_ba['gradur_ba']
df_ma['gradur'] = df_ma['gradur_ma']
df_final = pd.concat([df_ba, df_ma])

# Check the results
print("Bachelor's degrees:")
print(df_final[df_final['DGRDG']==1][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

print("\nMaster's degrees:")
print(df_final[df_final['DGRDG']==2][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

# Check sample sizes
print("\nSample sizes:")
print(df_final.groupby(['BAYR', 'DGRDG'])['gradur'].agg(['count', 'mean']).round(2))

In [ ]:
# Check for missing values before preparing regression variables
print("Missing values in original data:")
print(df_final[['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur','assigned_id']].isnull().sum())

# Clean the data
df_final = df_final.dropna(subset=['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur'])

# drop obs without assigned id (multiple matches within year)
df_final = df_final.dropna(subset=['assigned_id'])


In [ ]:
len(df_final)

In [ ]:
df_final.to_csv('df_final.csv', index = False)


In [ ]:
#### Now prepare regression variables 
#### ************************** Lisa Kahn’s set-up ***************************

import numpy as np
import seaborn as sns
import statsmodels.api as sm
# 1. Create log salary
df_final['log_salary'] = np.log(df_final['SALARY'])

# 2. Create experience variables
df_final['exp'] = df_final['REFYR'] - df_final['BAYR']
df_final['exp2'] = df_final['exp']**2

# 3. Create dummy for master's degree
df_final['master'] = (df_final['DGRDG'] == 2).astype(int)

# 4. Create interaction terms
df_final['gradur_master'] = df_final['gradur'] * df_final['master']
df_final['gradur_exp'] = df_final['gradur'] * df_final['exp']
df_final['gradur_master_exp'] = df_final['gradur'] * df_final['master'] * df_final['exp']

In [ ]:
# Check for any missing values in regression variables
reg_vars = ['log_salary', 'gradur', 'exp', 'exp2', 'master', 
            'gradur_master', 'gradur_exp', 'gradur_master_exp']
print("\nMissing values in regression variables:")
print(df_final[reg_vars].isnull().sum())

# 5. Create year dummies
year_dummies = pd.get_dummies(df_final['REFYR'], prefix='year')
base_year = min(df_final['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)
# # 6. Create individual fixed effects
# individual_dummies = pd.get_dummies(df_final['assigned_id'], prefix='id')

# 1. work on unique assigned ids (non-repeated respondents)

In [ ]:
# Get counts of each assigned_id
id_counts = df_final['assigned_id'].value_counts()

# Keep only observations where assigned_id appears once
df_unique = df_final[df_final['assigned_id'].isin(id_counts[id_counts == 1].index)]

In [ ]:
len(df_unique)

In [ ]:
# ## cross-validation: correct df_unique
# unique_keys = df_final.groupby('match_key').size()
# solo_keys = unique_keys[unique_keys == 1].index

# print(len(solo_keys))

In [ ]:
df_final.columns

### Lisa's Approach:  
log(salary)_it = α + β1(gradur)_c + β2(gradur)_c × exp_it + β3master_i + β4(gradur)_c × master_i + β5(gradur)_c × master_i × exp_it + δ1exp_it + δ2exp^2_it + γ_t + ε_it.

In [ ]:
# 1. Create log salary and key variables
df_unique['log_salary'] = np.log(df_unique['SALARY'])
df_unique['exp'] = df_unique['REFYR'] - df_unique['BAYR']
df_unique['exp2'] = df_unique['exp']**2
df_unique['master'] = (df_unique['DGRDG'] == 2).astype(int)

# 2. Create interaction terms 
df_unique['gradur_master'] = df_unique['gradur'] * df_unique['master']
df_unique['gradur_exp'] = df_unique['gradur'] * df_unique['exp']
df_unique['gradur_master_exp'] = df_unique['gradur'] * df_unique['master'] * df_unique['exp']

# 3. Create year dummies
year_dummies = pd.get_dummies(df_unique['REFYR'], prefix='year')
base_year = min(df_unique['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# 4. Combine variables for regression
X = pd.concat([
   df_unique[['gradur', 'gradur_exp', 'master', 'gradur_master', 'gradur_master_exp', 
             'exp', 'exp2']], 
   year_dummies
], axis=1)
X = sm.add_constant(X)

# 5. Run regression
model = sm.OLS(df_unique['log_salary'], X)
results = model.fit(cov_type='HC1')

# 6. Print results and interpretations
print(results.summary())
print("\nDetailed Interpretation of Key Coefficients:")
print("\n1. Initial effect of unemployment rate for bachelor's (β1):")
print(f"Coefficient: {results.params['gradur']:.4f}")
print(f"Standard Error: {results.bse['gradur']:.4f}")
print(f"P-value: {results.pvalues['gradur']:.4f}")

print("\n2. How unemployment effect changes with experience for bachelor's (β2):")
print(f"Coefficient: {results.params['gradur_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_exp']:.4f}")

print("\n3. Baseline master's premium (β3):")
print(f"Coefficient: {results.params['master']:.4f}")
print(f"Standard Error: {results.bse['master']:.4f}")
print(f"P-value: {results.pvalues['master']:.4f}")

print("\n4. Additional initial effect of unemployment for master's (β4):")
print(f"Coefficient: {results.params['gradur_master']:.4f}")
print(f"Standard Error: {results.bse['gradur_master']:.4f}")
print(f"P-value: {results.pvalues['gradur_master']:.4f}")

print("\n5. How master's unemployment effect changes with experience (β5):")
print(f"Coefficient: {results.params['gradur_master_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_master_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_master_exp']:.4f}")

# 7. Save regression results
df_results = pd.DataFrame({
   'coefficient': results.params,
   'std_err': results.bse,
   'p_value': results.pvalues,
   't_stat': results.tvalues
})
print("\nFull regression results saved in df_results")

In [ ]:
# Filter for experience between 3-15 years
df_unique_filtered_exp = df_unique[(df_unique['exp'] >= 3) & (df_unique['exp'] <= 15)]


# Create year dummies
year_dummies = pd.get_dummies(df_unique_filtered_exp['REFYR'], prefix='year')
base_year = min(df_unique_filtered_exp['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# 4. Combine variables for regression
X = pd.concat([
  df_unique_filtered_exp[['gradur', 'gradur_exp', 'master', 'gradur_master', 'gradur_master_exp', 
            'exp', 'exp2']], 
  year_dummies
], axis=1)
X = sm.add_constant(X)

# 5. Run regression
model = sm.OLS(df_unique_filtered_exp['log_salary'], X)
results = model.fit(cov_type='HC1')

# 6. Print results and interpretations
print(results.summary())
print("\nDetailed Interpretation of Key Coefficients:")
print("\n1. Initial effect of unemployment rate for bachelor's (β1):")
print(f"Coefficient: {results.params['gradur']:.4f}")
print(f"Standard Error: {results.bse['gradur']:.4f}")
print(f"P-value: {results.pvalues['gradur']:.4f}")

print("\n2. How unemployment effect changes with experience for bachelor's (β2):")
print(f"Coefficient: {results.params['gradur_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_exp']:.4f}")

print("\n3. Baseline master's premium (β3):")
print(f"Coefficient: {results.params['master']:.4f}")
print(f"Standard Error: {results.bse['master']:.4f}")
print(f"P-value: {results.pvalues['master']:.4f}")

print("\n4. Additional initial effect of unemployment for master's (β4):")
print(f"Coefficient: {results.params['gradur_master']:.4f}")
print(f"Standard Error: {results.bse['gradur_master']:.4f}")
print(f"P-value: {results.pvalues['gradur_master']:.4f}")

print("\n5. How master's unemployment effect changes with experience (β5):")
print(f"Coefficient: {results.params['gradur_master_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_master_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_master_exp']:.4f}")

# 7. Save regression results
df_results = pd.DataFrame({
  'coefficient': results.params,
  'std_err': results.bse,
  'p_value': results.pvalues,
  't_stat': results.tvalues
})
print("\nFull regression results saved in df_results")

In [ ]:
# Get 1st and 99th percentiles from filtered experience sample
p1 = df_unique_filtered_exp['SALARY'].quantile(0.01)
p99 = df_unique_filtered_exp['SALARY'].quantile(0.99)

# Filter for salary
df_unique_filtered_exp_salary = df_unique_filtered_exp[
   (df_unique_filtered_exp['SALARY'] >= p1) &
   (df_unique_filtered_exp['SALARY'] <= p99)
]

# Create year dummies
year_dummies = pd.get_dummies(df_unique_filtered_exp_salary['REFYR'], prefix='year')
base_year = min(df_unique_filtered_exp_salary['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Regression setup and run
X = pd.concat([
  df_unique_filtered_exp_salary[['gradur', 'gradur_exp', 'master', 'gradur_master', 
                                'gradur_master_exp', 'exp', 'exp2']], 
  year_dummies
], axis=1)
X = sm.add_constant(X)

model = sm.OLS(df_unique_filtered_exp_salary['log_salary'], X)
results = model.fit(cov_type='HC1')
print(results.summary())

# Save results
df_results = pd.DataFrame({
  'coefficient': results.params,
  'std_err': results.bse,
  'p_value': results.pvalues,
  't_stat': results.tvalues
})

## Oreopolous' Approach

In [ ]:
# Create unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in df_unique['exp'].unique():
  ur_exp_interactions[f'ur_exp_{exp}'] = df_unique['gradur'] * (df_unique['exp'] == exp)

# Create master's degree interactions with experience  
master_exp_interactions = pd.DataFrame()
for exp in df_unique['exp'].unique():
  master_exp_interactions[f'master_exp_{exp}'] = df_unique['master'] * (df_unique['exp'] == exp)

# Create master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in df_unique['exp'].unique():
  master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_unique['gradur'] * df_unique['master'] * (df_unique['exp'] == exp)

# Create year dummies
year_dummies = pd.get_dummies(df_unique['REFYR'], prefix='year')
base_year = min(df_unique['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
  ur_exp_interactions,  
  master_exp_interactions,  
  master_ur_exp_interactions,  
  year_dummies  
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_unique['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print effects by experience year
print("\nEffects by experience year:")
for exp in sorted(df_unique['exp'].unique()):
  print(f"\nExperience {exp} years:")
  
  # Bachelor's unemployment effect
  ur_coef = f'ur_exp_{exp}'
  if ur_coef in results.params:
      print(f"Bachelor's unemployment effect:")
      print(f"Coefficient: {results.params[ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[ur_coef]:.4f}")
  
  # Master's premium
  master_coef = f'master_exp_{exp}'
  if master_coef in results.params:
      print(f"Master's premium:")
      print(f"Coefficient: {results.params[master_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_coef]:.4f}")
  
  # Additional unemployment effect for master's 
  master_ur_coef = f'master_ur_exp_{exp}'
  if master_ur_coef in results.params:
      print(f"Additional master's unemployment effect:")
      print(f"Coefficient: {results.params[master_ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_ur_coef]:.4f}")

# restrict sample size to experience year [3-15]


In [ ]:
# Create unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp['exp'].unique():
  ur_exp_interactions[f'ur_exp_{exp}'] = df_unique_filtered_exp['gradur'] * (df_unique_filtered_exp['exp'] == exp)

# Create master's degree interactions with experience  
master_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp['exp'].unique():
  master_exp_interactions[f'master_exp_{exp}'] = df_unique_filtered_exp['master'] * (df_unique_filtered_exp['exp'] == exp)

# Create master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp['exp'].unique():
  master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_unique_filtered_exp['gradur'] * df_unique_filtered_exp['master'] * (df_unique_filtered_exp['exp'] == exp)

# Create year dummies
year_dummies = pd.get_dummies(df_unique_filtered_exp['REFYR'], prefix='year')
base_year = min(df_unique_filtered_exp['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
  ur_exp_interactions,  
  master_exp_interactions,  
  master_ur_exp_interactions,  
  year_dummies  
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_unique_filtered_exp['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print effects by experience year
print("\nEffects by experience year:")
for exp in sorted(df_unique_filtered_exp['exp'].unique()):
  print(f"\nExperience {exp} years:")
  
  # Bachelor's unemployment effect
  ur_coef = f'ur_exp_{exp}'
  if ur_coef in results.params:
      print(f"Bachelor's unemployment effect:")
      print(f"Coefficient: {results.params[ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[ur_coef]:.4f}")
  
  # Master's premium
  master_coef = f'master_exp_{exp}'
  if master_coef in results.params:
      print(f"Master's premium:")
      print(f"Coefficient: {results.params[master_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_coef]:.4f}")
  
  # Additional unemployment effect for master's 
  master_ur_coef = f'master_ur_exp_{exp}'
  if master_ur_coef in results.params:
      print(f"Additional master's unemployment effect:")
      print(f"Coefficient: {results.params[master_ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_ur_coef]:.4f}")

## ***remove outliers (salary outside 1%-99%)

In [ ]:
# Create unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp_salary['exp'].unique():
  ur_exp_interactions[f'ur_exp_{exp}'] = df_unique_filtered_exp_salary['gradur'] * (df_unique_filtered_exp_salary['exp'] == exp)

# Create master's degree interactions with experience  
master_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp_salary['exp'].unique():
  master_exp_interactions[f'master_exp_{exp}'] = df_unique_filtered_exp_salary['master'] * (df_unique_filtered_exp_salary['exp'] == exp)

# Create master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in df_unique_filtered_exp_salary['exp'].unique():
  master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_unique_filtered_exp_salary['gradur'] * df_unique_filtered_exp_salary['master'] * (df_unique_filtered_exp_salary['exp'] == exp)

# Create year dummies
year_dummies = pd.get_dummies(df_unique_filtered_exp_salary['REFYR'], prefix='year')
base_year = min(df_unique_filtered_exp_salary['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
  ur_exp_interactions,  
  master_exp_interactions,  
  master_ur_exp_interactions,  
  year_dummies  
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_unique_filtered_exp_salary['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print effects by experience year
print("\nEffects by experience year:")
for exp in sorted(df_unique_filtered_exp_salary['exp'].unique()):
  print(f"\nExperience {exp} years:")
  
  # Bachelor's unemployment effect
  ur_coef = f'ur_exp_{exp}'
  if ur_coef in results.params:
      print(f"Bachelor's unemployment effect:")
      print(f"Coefficient: {results.params[ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[ur_coef]:.4f}")
  
  # Master's premium  
  master_coef = f'master_exp_{exp}'
  if master_coef in results.params:
      print(f"Master's premium:")
      print(f"Coefficient: {results.params[master_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_coef]:.4f}")
  
  # Additional unemployment effect for master's 
  master_ur_coef = f'master_ur_exp_{exp}'
  if master_ur_coef in results.params:
      print(f"Additional master's unemployment effect:")
      print(f"Coefficient: {results.params[master_ur_coef]:.4f}")
      print(f"Standard Error: {results.bse[master_ur_coef]:.4f}")
      print(f"P-value: {results.pvalues[master_ur_coef]:.4f}")